# PrediRuta: carga de las tablas depuradas a Supabase


Este notebook toma los archivos CSV generados en `limpieza_estructuracion_bd.ipynb`, crea la estructura relacional en Supabase y carga la información mediante una conexión directa con PostgreSQL. Las credenciales necesarias se leen desde el archivo `.env` para evitar incluirlas directamente en el código.

Las tablas se cargan en un orden que permite mantener correctamente sus relaciones:

- `CLIMA`: contiene las observaciones meteorológicas utilizadas por los accidentes.
- `ACCIDENTE`: contiene una fila por accidente utilizable y su relación con la información climática.
- `VIA`: conserva una vía representativa para cada accidente.
- `VEHICULO`: contiene los vehículos asociados a los accidentes.
- `ACTOR_VIAL`: reúne los actores involucrados en cada accidente.
- `CAUSA`: contiene las causas registradas y su relación con los accidentes o vehículos.

## 0. Configuración y conexión


En esta etapa se preparan las dependencias necesarias, se define la ubicación de los archivos CSV y se cargan los datos de conexión almacenados en `.env`. El notebook reconoce las variables `host`, `port`, `database`, `user` y `password`, junto con sus equivalentes con prefijo `SUPABASE_`.

La contraseña se utiliza únicamente para establecer la conexión con la base de datos y no se muestra durante la ejecución.

In [1]:
# Importar librerías necesarias
from pathlib import Path
from io import StringIO
import os
import time
import pandas as pd
import psycopg
from psycopg import sql
from dotenv import load_dotenv
from IPython.display import display


In [2]:
# Cargar las variables de conexión almacenadas en .env
load_dotenv()

# Configuración de rutas y datos de conexión
CARPETA_CSV = Path('data_procesada/tablas_finales')

SUPABASE_HOST = os.getenv('host') or os.getenv('SUPABASE_HOST')
SUPABASE_PORT = int(os.getenv('port') or os.getenv('SUPABASE_PORT', '5432'))
SUPABASE_USER = os.getenv('user') or os.getenv('SUPABASE_USER', 'postgres')
SUPABASE_DB = os.getenv('database') or os.getenv('SUPABASE_DB', 'postgres')
SUPABASE_PASS = os.getenv('password') or os.getenv('SUPABASE_PASS')

print('Configuración de conexión cargada correctamente.')

Configuración de conexión cargada correctamente.


## 1. Revisión de los archivos CSV


Antes de crear las tablas se verifica que los seis archivos CSV estén disponibles en `data_procesada/tablas_finales` y que sus columnas coincidan con la estructura definida para cada tabla. Esta revisión permite detectar archivos faltantes o cambios en los encabezados antes de iniciar la carga de información a Supabase.

In [3]:
# Definición de archivos y columnas esperadas
ARCHIVOS = {
    'clima': 'CLIMA.csv',
    'accidente': 'ACCIDENTE.csv',
    'via': 'VIA.csv',
    'vehiculo': 'VEHICULO.csv',
    'actor_vial': 'ACTOR_VIAL.csv',
    'causa': 'CAUSA.csv',
}

COLUMNAS = {
    'clima': ['CLIMA_ID','FECHA_HORA_CLIMA','LATITUD_CELDA','LONGITUD_CELDA','LATITUD_MODELO','LONGITUD_MODELO','ELEVACION_MODELO_M','TEMPERATURA_2M','HUMEDAD_RELATIVA_2M','SENSACION_TERMICA','PRECIPITACION','LLUVIA','CODIGO_CLIMA','NUBOSIDAD','PRESION_SUPERFICIE','VELOCIDAD_VIENTO_10M','DIRECCION_VIENTO_10M','MODELO','FUENTE'],
    'accidente': ['ACCIDENTE_ID','FORMULARIO','FECHA_HORA','LATITUD','LONGITUD','DIRECCION','GRAVEDAD','CLASE_ACCIDENTE','LOCALIDAD','BARRIO','CIV','PK_CALZADA','VIA_CERCANA','DISTANCIA_VIA','CLIMA_ID','ANIO','MES','DIA_SEMANA','HORA','ES_FIN_SEMANA','OBJETIVO_GRAVE'],
    'via': ['ACCIDENTE_ID','CODIGO_VIA','GEOMETRIA_PLANTA','GEOMETRIA_TERRENO','GEOMETRIA_SECCION','SENTIDO_VIA','N_CALZADAS','N_CARRILES','SUPERFICIE_RODADURA','ESTADO_VIA','CONDICION_VIA','ILUMINACION_ARTIFICIAL','SEMAFORO','VISIBILIDAD_VIA'],
    'vehiculo': ['ACCIDENTE_ID','CODIGO_VEHICULO','CLASE','SERVICIO'],
    'actor_vial': ['ACCIDENTE_ID','CODIGO_ACTOR','CODIGO_VEHICULO','CONDICION','ESTADO','GENERO','EDAD'],
    'causa': ['ACCIDENTE_ID','CODIGO_VEHICULO','CODIGO_CAUSA','NOMBRE','TIPO'],
}

In [4]:
# Validar archivos CSV y sus columnas
for tabla, archivo in ARCHIVOS.items():
    ruta = CARPETA_CSV / archivo

    columnas = pd.read_csv(ruta, nrows=0, encoding='utf-8-sig').columns.tolist()

    if columnas != COLUMNAS[tabla]:
        raise ValueError(f'Las columnas de {archivo} no coinciden con las esperadas.')

print('Archivos CSV y columnas validados correctamente.')

Archivos CSV y columnas validados correctamente.


## 2. Prueba de conexión con Supabase


La conexión se realiza con las credenciales almacenadas en `.env` y utilizando SSL. Antes de realizar cambios en la base de datos, se consulta la versión de PostgreSQL para verificar que la conexión con Supabase se estableció correctamente.

In [5]:
# Configurar y probar la conexión con Supabase
CONEXION = {
    'host': SUPABASE_HOST,
    'port': SUPABASE_PORT,
    'dbname': SUPABASE_DB,
    'user': SUPABASE_USER,
    'password': SUPABASE_PASS,
    'sslmode': 'require',
    'connect_timeout': 20,
}

with psycopg.connect(**CONEXION) as conexion:
    version = conexion.execute('SELECT version()').fetchone()[0]

print('Conexión correcta:', version.split(',')[0])

OperationalError: connection failed: connection to server at "44.238.118.41", port 5432 failed: FATAL:  the database system is not accepting connections
DETAIL:  Hot standby mode is disabled.
Multiple connection attempts failed. All failures were:
- host: 'aws-0-us-west-2.pooler.supabase.com', port: 5432, hostaddr: '35.160.209.8': connection failed: connection to server at "35.160.209.8", port 5432 failed: FATAL:  the database system is not accepting connections
DETAIL:  Hot standby mode is disabled.
- host: 'aws-0-us-west-2.pooler.supabase.com', port: 5432, hostaddr: '54.70.143.232': connection failed: connection to server at "54.70.143.232", port 5432 failed: FATAL:  the database system is not accepting connections
DETAIL:  Hot standby mode is disabled.
- host: 'aws-0-us-west-2.pooler.supabase.com', port: 5432, hostaddr: '44.238.118.41': connection failed: connection to server at "44.238.118.41", port 5432 failed: FATAL:  the database system is not accepting connections
DETAIL:  Hot 

## 3. Creación de las tablas y relaciones


La estructura de la base se crea a partir de las columnas definidas en los archivos CSV finales. Primero se crea `CLIMA`, ya que `ACCIDENTE` puede tener una referencia a una observación meteorológica. Después se crean `VIA`, `VEHICULO`, `ACTOR_VIAL` y `CAUSA`, relacionadas con `ACCIDENTE` mediante `ACCIDENTE_ID`.

Los nombres de las tablas y columnas se manejan en minúscula siguiendo la convención de PostgreSQL. Si una tabla ya existe en Supabase, se mantiene y no se elimina durante este proceso.

In [ ]:
# Crear las tablas y relaciones del modelo final en PostgreSQL

DDL = {
    'clima': '''CREATE TABLE IF NOT EXISTS clima (
        clima_id text PRIMARY KEY,
        fecha_hora_clima timestamp NOT NULL,
        latitud_celda double precision,
        longitud_celda double precision,
        latitud_modelo double precision,
        longitud_modelo double precision,
        elevacion_modelo_m double precision,
        temperatura_2m double precision,
        humedad_relativa_2m double precision,
        sensacion_termica double precision,
        precipitacion double precision,
        lluvia double precision,
        codigo_clima integer,
        nubosidad double precision,
        presion_superficie double precision,
        velocidad_viento_10m double precision,
        direccion_viento_10m double precision,
        modelo text NOT NULL,
        fuente text
    )''',
    'accidente': '''CREATE TABLE IF NOT EXISTS accidente (
        accidente_id text PRIMARY KEY,
        formulario text NOT NULL UNIQUE,
        fecha_hora timestamp NOT NULL,
        latitud double precision NOT NULL,
        longitud double precision NOT NULL,
        direccion text,
        gravedad text,
        clase_accidente text,
        localidad text,
        barrio text,
        civ text,
        pk_calzada text,
        via_cercana text,
        distancia_via double precision,
        clima_id text REFERENCES clima(clima_id),
        anio integer NOT NULL,
        mes integer NOT NULL,
        dia_semana integer NOT NULL,
        hora integer NOT NULL,
        es_fin_semana boolean NOT NULL,
        objetivo_grave boolean NOT NULL
    )''',
    'via': '''CREATE TABLE IF NOT EXISTS via (
        accidente_id text PRIMARY KEY REFERENCES accidente(accidente_id),
        codigo_via text,
        geometria_planta text,
        geometria_terreno text,
        geometria_seccion text,
        sentido_via text,
        n_calzadas numeric,
        n_carriles numeric,
        superficie_rodadura text,
        estado_via text,
        condicion_via text,
        iluminacion_artificial text,
        semaforo text,
        visibilidad_via text
    )''',
    'vehiculo': '''CREATE TABLE IF NOT EXISTS vehiculo (
        accidente_id text NOT NULL REFERENCES accidente(accidente_id),
        codigo_vehiculo text NOT NULL,
        clase text,
        servicio text,
        PRIMARY KEY (accidente_id, codigo_vehiculo)
    )''',
    'actor_vial': '''CREATE TABLE IF NOT EXISTS actor_vial (
        accidente_id text NOT NULL REFERENCES accidente(accidente_id),
        codigo_actor text NOT NULL,
        codigo_vehiculo text,
        condicion text,
        estado text,
        genero text,
        edad numeric,
        PRIMARY KEY (accidente_id, codigo_actor)
    )''',
    'causa': '''CREATE TABLE IF NOT EXISTS causa (
        accidente_id text NOT NULL REFERENCES accidente(accidente_id),
        codigo_vehiculo text,
        codigo_causa text NOT NULL,
        nombre text,
        tipo text
    )''',
}

with psycopg.connect(**CONEXION) as conexion:
    for sentencia in DDL.values():
        conexion.execute(sentencia)

    # Ajustar tablas creadas en una ejecución anterior con tipos incompatibles con los CSV.
    conexion.execute('''ALTER TABLE via
        ALTER COLUMN n_calzadas TYPE numeric USING n_calzadas::numeric,
        ALTER COLUMN n_carriles TYPE numeric USING n_carriles::numeric''')
    conexion.execute('''ALTER TABLE actor_vial
        ALTER COLUMN edad TYPE numeric USING edad::numeric''')
    conexion.execute('''CREATE UNIQUE INDEX IF NOT EXISTS causa_llave_unica
        ON causa (accidente_id, codigo_vehiculo, codigo_causa) NULLS NOT DISTINCT''')

print('Estructura de la base creada correctamente en Supabase.')


## 4. Revisión del estado actual de las tablas


Antes de iniciar la carga se revisa cuántos registros existen actualmente en cada tabla de Supabase. Si una tabla ya contiene información, se omite para evitar duplicar o sobrescribir datos.

Si alguna carga anterior quedó incompleta, primero se debe revisar su contenido antes de decidir si es necesario eliminarla y volver a cargarla.

In [ ]:
# Revisar la cantidad de registros existentes en cada tabla
with psycopg.connect(**CONEXION) as conexion:
    estado_inicial = pd.DataFrame([
        {
            'tabla': tabla,
            'filas_en_supabase': conexion.execute(
                sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
            ).fetchone()[0]
        }
        for tabla in ARCHIVOS
    ])

display(estado_inicial)


## 5. Carga de los archivos CSV


Los archivos se cargan en lotes de `50.000` filas y respetando el orden de las relaciones. Cada lote se copia primero a una tabla temporal y luego se incorpora a la tabla definitiva sin repetir llaves existentes. Esta estrategia reduce el espacio transitorio requerido por PostgreSQL y permite reanudar una tabla parcialmente cargada. Durante la ejecución se muestra el avance y las tablas que ya coinciden con su CSV se omiten.


In [ ]:
# Cargar los CSV por lotes en el orden definido por las relaciones
ORDEN_CARGA = ['clima', 'accidente', 'via', 'vehiculo', 'actor_vial', 'causa']
FILAS_POR_LOTE = 50_000
resultado_carga = []

def contar_filas_archivo(ruta):
    with ruta.open('rb') as archivo:
        return max(sum(1 for _ in archivo) - 1, 0)

with psycopg.connect(**CONEXION) as conexion:
    for numero, tabla in enumerate(ORDEN_CARGA, 1):
        ruta = CARPETA_CSV / ARCHIVOS[tabla]
        filas_csv = contar_filas_archivo(ruta)
        existentes = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]

        if existentes == filas_csv:
            print(f'{numero}/{len(ORDEN_CARGA)} {tabla}: omitida ({existentes:,} filas completas)')
            resultado_carga.append([tabla, 'omitida', existentes, 0])
            continue

        columnas = [columna.lower() for columna in COLUMNAS[tabla]]
        temporal = f'_carga_{tabla}'
        conexion.execute(
            sql.SQL('CREATE TEMP TABLE {} (LIKE {} INCLUDING DEFAULTS) ON COMMIT DELETE ROWS').format(
                sql.Identifier(temporal),
                sql.Identifier(tabla),
            )
        )

        consulta_copy = sql.SQL(
            'COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE, ENCODING UTF8)'
        ).format(
            sql.Identifier(temporal),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
        )
        consulta_insertar = sql.SQL(
            'INSERT INTO {} ({}) SELECT {} FROM {} ON CONFLICT DO NOTHING'
        ).format(
            sql.Identifier(tabla),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
            sql.Identifier(temporal),
        )

        inicio = time.time()
        for lote, bloque in enumerate(pd.read_csv(
            ruta,
            dtype='string',
            encoding='utf-8-sig',
            chunksize=FILAS_POR_LOTE,
            low_memory=False,
        ), 1):
            buffer = StringIO()
            bloque.to_csv(buffer, index=False)
            buffer.seek(0)

            with conexion.cursor().copy(consulta_copy) as copia:
                copia.write(buffer.getvalue())
            conexion.execute(consulta_insertar)
            conexion.commit()

            cargadas = conexion.execute(
                sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
            ).fetchone()[0]
            print(
                f'\r{numero}/{len(ORDEN_CARGA)} {tabla}: '
                f'{cargadas:,} / {filas_csv:,}',
                end='',
                flush=True,
            )

        tiempo = round(time.time() - inicio, 1)
        cargadas = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]
        print(f' en {tiempo} s')
        resultado_carga.append([tabla, 'completa' if cargadas == filas_csv else 'incompleta', cargadas, tiempo])

display(pd.DataFrame(
    resultado_carga,
    columns=['tabla', 'resultado', 'filas_en_supabase', 'segundos'],
))


## 6. Validación de la carga


Finalmente se compara la cantidad de registros de cada CSV con los almacenados en Supabase. También se revisa que las tablas relacionadas con `ACCIDENTE` no tengan referencias huérfanas y que los `CLIMA_ID` utilizados existan en la tabla `CLIMA`.

La carga se considera completa cuando las cantidades coinciden entre los archivos y la base de datos, y no se encuentran relaciones inválidas entre las tablas.

In [ ]:
# Validar cantidades de registros y relaciones entre tablas
def contar_filas_csv(ruta):
    with ruta.open('rb') as archivo:
        return max(sum(bloque.count(b'\n') for bloque in iter(
            lambda: archivo.read(1024 * 1024), b''
        )) - 1, 0)

# Validar cantidades de registros y relaciones entre tablas
with psycopg.connect(**CONEXION) as conexion:

    # Comparar cantidad de filas entre los CSV y Supabase
    cantidades = []

    for tabla, archivo in ARCHIVOS.items():
        filas_csv = contar_filas_csv(CARPETA_CSV / archivo)
        filas_bd = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]

        cantidades.append([tabla, filas_csv, filas_bd, filas_csv == filas_bd])

    # Validar referencias hacia ACCIDENTE
    huerfanos = {}

    for tabla in ['via', 'vehiculo', 'actor_vial', 'causa']:
        huerfanos[tabla] = conexion.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM {} t
                LEFT JOIN accidente a USING (accidente_id)
                WHERE a.accidente_id IS NULL
            ''').format(sql.Identifier(tabla))
        ).fetchone()[0]

    # Validar referencias de ACCIDENTE hacia CLIMA
    huerfanos['accidente_clima'] = conexion.execute('''
        SELECT COUNT(*)
        FROM accidente a
        LEFT JOIN clima c USING (clima_id)
        WHERE a.clima_id IS NOT NULL
          AND c.clima_id IS NULL
    ''').fetchone()[0]

display(pd.DataFrame(
    cantidades,
    columns=['tabla', 'filas_csv', 'filas_supabase', 'coincide']
))

display(pd.DataFrame(
    huerfanos.items(),
    columns=['relacion', 'referencias_huerfanas']
))
